1° step - estrazione dei dati relativi alla statistica degli incidenti stradali tramite api ISTAT


In [3]:
## importiamo le librerie 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [6]:
import requests

Prima di scaricare i dati, controllo che il dataset risponda e ne leggo la struttura

In [7]:
url_struct = 'https://esploradati.istat.it/SDMXWS/rest/dataflow/IT1/41_983/latest'
r = requests.get(url_struct, verify=False, timeout=60)
print(r.status_code)
print(r.text[:1000])

200
<?xml version="1.0" encoding="utf-8"?>
<!--NSI Web Service v9.11.0.0-->
<message:Structure xmlns:message="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/message" xmlns:structure="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/structure" xmlns:common="http://www.sdmx.org/resources/sdmxml/schemas/v2_1/common">
  <message:Header>
    <message:ID>IDREF6632</message:ID>
    <message:Test>false</message:Test>
    <message:Prepared>2026-10-13T22:47:23.2533809+02:00</message:Prepared>
    <message:Sender id="Unknown" />
    <message:Receiver id="Unknown" />
  </message:Header>
  <message:Structures>
    <structure:Dataflows>
      <structure:Dataflow id="41_983" agencyID="IT1" version="1.0" isFinal="true">
        <common:Annotations>
          <common:Annotation id="DDBDataflow">
            <common:AnnotationTitle>851859858F7F55E81C027BB3764D67E3</common:AnnotationTitle>
            <common:AnnotationType>DDBDataflow</common:AnnotationType>
          </common:Annotatio


In [8]:
url = 'https://esploradati.istat.it/SDMXWS/rest/data/41_983'

In [10]:
r = requests.get(url, headers={'Accept': 'application/vnd.sdmx.data+csv;version=1.0.0'}, verify=False, timeout=60)
type(r)


ReadTimeout: HTTPSConnectionPool(host='esploradati.istat.it', port=443): Read timed out. (read timeout=60)

In [ ]:
r.status_code

200

trasformo in tabella e salvo , stringIO mi serve per trasformare una stringa di testo in un file che read csv possa leggere, poi lo salvo come csv

In [3]:
from io import StringIO

df = pd.read_csv(StringIO(r.text))
print(df.shape)
df.head(20)

NameError: name 'r' is not defined

In [4]:
df.to_csv("dati_istat.csv", index=False)

NameError: name 'df' is not defined

In [4]:
df = pd.read_csv("dati_istat.csv")
df.head()

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
co = pd.read_csv("Comuni.csv")
print(co.columns.tolist())

['Codice Ripartizione geografica;Codice Regione;Codice Provincia (Storico);Codice Provincia/Uts;Codice Comune (alfanumerico);Codice Comune (numerico);Comune;Comune (dizione straniera);Sigla automobilistica;Capoluogo di Provincia/Uts;Capoluogo di Regione;Popolazione legale;Anno Censimento;Superficie (Kmq);Anno (Superficie);Popolazione residente;Anno (Popolazione residente)']


In [6]:
co = pd.read_csv(r"C:\Users\danub\Desktop\Visual Data\my project\Comuni.csv", sep=";", encoding="latin-1")
print(co.columns.tolist())

['ï»¿Codice Ripartizione geografica', 'Codice Regione', 'Codice Provincia (Storico)', 'Codice Provincia/Uts', 'Codice Comune (alfanumerico)', 'Codice Comune (numerico)', 'Comune', 'Comune (dizione straniera)', 'Sigla automobilistica', 'Capoluogo di Provincia/Uts', 'Capoluogo di Regione', 'Popolazione legale', 'Anno Censimento', 'Superficie (Kmq)', 'Anno (Superficie)', 'Popolazione residente', 'Anno (Popolazione residente)']


Unisco i due file per riportarmi il nome del comune

In [7]:
df_incidenti = df.merge(co[['Codice Comune (numerico)','Comune']], how='left', left_on='REF_AREA', right_on='Codice Comune (numerico)')
df_incidenti.head()

,DATAFLOW,FREQ,REF_AREA,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT,Codice Comune (numerico),Comune
0,IT1:41_983(1.0),A,1001,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001.0,AgliÃ¨
1,IT1:41_983(1.0),A,1001,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001.0,AgliÃ¨
2,IT1:41_983(1.0),A,1001,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001.0,AgliÃ¨
3,IT1:41_983(1.0),A,1001,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001.0,AgliÃ¨
4,IT1:41_983(1.0),A,1001,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001.0,AgliÃ¨


Sposto la colonna comune

In [8]:
col = df_incidenti.pop("Comune")
df_incidenti.insert(3,"Comune", col)
df_incidenti.head()

,DATAFLOW,FREQ,REF_AREA,Comune,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE,OBS_STATUS,NOTE_DS,NOTE_REF_AREA,NOTE_DATA_TYPE,NOTE_RESULT,NOTE_TIME_PERIOD,BASE_PER,UNIT_MEAS,UNIT_MULT,Codice Comune (numerico)
0,IT1:41_983(1.0),A,1001,AgliÃ¨,KILLINJ,F,2001,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001.0
1,IT1:41_983(1.0),A,1001,AgliÃ¨,KILLINJ,F,2002,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001.0
2,IT1:41_983(1.0),A,1001,AgliÃ¨,KILLINJ,F,2003,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001.0
3,IT1:41_983(1.0),A,1001,AgliÃ¨,KILLINJ,F,2004,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001.0
4,IT1:41_983(1.0),A,1001,AgliÃ¨,KILLINJ,F,2005,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001.0


Verifico le colonne vuote per eliminarle

In [9]:
df_incidenti.isna().all()

DATAFLOW                    False
FREQ                        False
REF_AREA                    False
Comune                      False
DATA_TYPE                   False
RESULT                      False
TIME_PERIOD                 False
OBS_VALUE                   False
OBS_STATUS                   True
NOTE_DS                      True
NOTE_REF_AREA                True
NOTE_DATA_TYPE               True
NOTE_RESULT                  True
NOTE_TIME_PERIOD             True
BASE_PER                     True
UNIT_MEAS                    True
UNIT_MULT                    True
Codice Comune (numerico)    False
dtype: bool

droppo le colonne che non hanno dati

In [10]:
df_incidenti = df_incidenti.drop(columns=["OBS_STATUS","NOTE_DS","NOTE_REF_AREA","NOTE_DATA_TYPE","NOTE_RESULT","NOTE_TIME_PERIOD","BASE_PER","UNIT_MEAS","UNIT_MULT","Codice Comune (numerico)"])
df_incidenti.head() 

,DATAFLOW,FREQ,REF_AREA,Comune,DATA_TYPE,RESULT,TIME_PERIOD,OBS_VALUE
0,IT1:41_983(1.0),A,1001,AgliÃ¨,KILLINJ,F,2001,10
1,IT1:41_983(1.0),A,1001,AgliÃ¨,KILLINJ,F,2002,10
2,IT1:41_983(1.0),A,1001,AgliÃ¨,KILLINJ,F,2003,7
3,IT1:41_983(1.0),A,1001,AgliÃ¨,KILLINJ,F,2004,13
4,IT1:41_983(1.0),A,1001,AgliÃ¨,KILLINJ,F,2005,2


Rendo leggibile la tabella trasformando i valori in valori leggibili

In [11]:
df_incidenti["MISURA"] = (df_incidenti["DATA_TYPE"] + "_" + df_incidenti["RESULT"]).map({
    "ROADACC_9": "incidenti",
    "KILLINJ_M": "morti",
    "KILLINJ_F": "feriti",
})

wide = df_incidenti.pivot_table(
    index=["REF_AREA", "Comune", "TIME_PERIOD"],
    columns="MISURA",
    values="OBS_VALUE"
).reset_index()

wide[["feriti", "incidenti", "morti"]] = wide[["feriti", "incidenti", "morti"]].astype("Int64")

In [12]:
wide.head()

MISURA,REF_AREA,Comune,TIME_PERIOD,feriti,incidenti,morti
0,1001,AgliÃ¨,2001,10,5,0
1,1001,AgliÃ¨,2002,10,5,0
2,1001,AgliÃ¨,2003,7,4,0
3,1001,AgliÃ¨,2004,13,9,0
4,1001,AgliÃ¨,2005,2,2,0


trasformo il dato della superficie da testo a numero e calcolo la densitaà della popolazione 

In [13]:
co["Superficie (Kmq)"] = (co["Superficie (Kmq)"]
                          .str.replace(".", "", regex=False)    # toglie le migliaia
                          .str.replace(",", ".", regex=False)   # virgola -> punto decimale
                          .astype(float))

co["densita"] = co["Popolazione residente"] / co["Superficie (Kmq)"]

In [14]:
co.head()

,ï»¿Codice Ripartizione geografica,Codice Regione,Codice Provincia (Storico),Codice Provincia/Uts,Codice Comune (alfanumerico),Codice Comune (numerico),Comune,Comune (dizione straniera),Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Popolazione legale,Anno Censimento,Superficie (Kmq),Anno (Superficie),Popolazione residente,Anno (Popolazione residente),densita
0,1,1,1,201,1001,1001,AgliÃ¨,NaN,TO,0,0,2562,2021,13.1463,2026,2585,2024,196.633273
1,1,1,1,201,1002,1002,Airasca,NaN,TO,0,0,3660,2021,15.7393,2026,3695,2024,234.762664
2,1,1,1,201,1003,1003,Ala di Stura,NaN,TO,0,0,467,2021,46.3316,2026,463,2024,9.993180
3,1,1,1,201,1004,1004,Albiano d'Ivrea,NaN,TO,0,0,1637,2021,11.7397,2026,1624,2024,138.334029
4,1,1,1,201,1006,1006,Almese,NaN,TO,0,0,6331,2021,17.8741,2026,6297,2024,352.297458


Unisco il datase dell'istat con il datatset con l'identificativo dei comuni

In [21]:
cols = ["Codice Comune (numerico)", "Popolazione residente", "densita",
        "Codice Regione", "Codice Provincia/Uts", "Sigla automobilistica",
        "Capoluogo di Provincia/Uts", "Capoluogo di Regione","Superficie (Kmq)"]

m = wide.merge(co[cols], left_on="REF_AREA", right_on="Codice Comune (numerico)",
               how="left", indicator=True).drop(columns="Codice Comune (numerico)")
print(m["_merge"].value_counts())

_merge
both          175813
left_only          0
right_only         0
Name: count, dtype: int64


In [22]:
m.head()

,REF_AREA,Comune,TIME_PERIOD,feriti,incidenti,morti,Popolazione residente,densita,Codice Regione,Codice Provincia/Uts,Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Superficie (Kmq),_merge
0,1001,AgliÃ¨,2001,10,5,0,2585,196.633273,1,201,TO,0,0,13.1463,both
1,1001,AgliÃ¨,2002,10,5,0,2585,196.633273,1,201,TO,0,0,13.1463,both
2,1001,AgliÃ¨,2003,7,4,0,2585,196.633273,1,201,TO,0,0,13.1463,both
3,1001,AgliÃ¨,2004,13,9,0,2585,196.633273,1,201,TO,0,0,13.1463,both
4,1001,AgliÃ¨,2005,2,2,0,2585,196.633273,1,201,TO,0,0,13.1463,both


Inserisco i ati per l'analisi

In [23]:
m["morti_100k"]  = m["morti"]  / m["Popolazione residente"] * 100_000
m["feriti_100k"] = m["feriti"] / m["Popolazione residente"] * 100_000
m["incidenti_100k"] = m["incidenti"] / m["Popolazione residente"] * 100_000
m["indice_mortalita"] = m["morti"] / m["incidenti"] * 1000   
m["incidenti_per_kmq"] = m["incidenti"] / m["Superficie (Kmq)"]

In [24]:
m.head()

,REF_AREA,Comune,TIME_PERIOD,feriti,incidenti,morti,Popolazione residente,densita,Codice Regione,Codice Provincia/Uts,Sigla automobilistica,Capoluogo di Provincia/Uts,Capoluogo di Regione,Superficie (Kmq),_merge,morti_100k,feriti_100k,incidenti_100k,indice_mortalita,incidenti_per_kmq
0,1001,AgliÃ¨,2001,10,5,0,2585,196.633273,1,201,TO,0,0,13.1463,both,0.0,386.847195,193.423598,0.0,0.380335
1,1001,AgliÃ¨,2002,10,5,0,2585,196.633273,1,201,TO,0,0,13.1463,both,0.0,386.847195,193.423598,0.0,0.380335
2,1001,AgliÃ¨,2003,7,4,0,2585,196.633273,1,201,TO,0,0,13.1463,both,0.0,270.793037,154.738878,0.0,0.304268
3,1001,AgliÃ¨,2004,13,9,0,2585,196.633273,1,201,TO,0,0,13.1463,both,0.0,502.901354,348.162476,0.0,0.684603
4,1001,AgliÃ¨,2005,2,2,0,2585,196.633273,1,201,TO,0,0,13.1463,both,0.0,77.369439,77.369439,0.0,0.152134


droppo le colonne che non servono

In [25]:
m = m.drop(columns=['Capoluogo di Provincia/Uts', 'Capoluogo di Regione'])
m.head()

,REF_AREA,Comune,TIME_PERIOD,feriti,incidenti,morti,Popolazione residente,densita,Codice Regione,Codice Provincia/Uts,Sigla automobilistica,Superficie (Kmq),_merge,morti_100k,feriti_100k,incidenti_100k,indice_mortalita,incidenti_per_kmq
0,1001,AgliÃ¨,2001,10,5,0,2585,196.633273,1,201,TO,13.1463,both,0.0,386.847195,193.423598,0.0,0.380335
1,1001,AgliÃ¨,2002,10,5,0,2585,196.633273,1,201,TO,13.1463,both,0.0,386.847195,193.423598,0.0,0.380335
2,1001,AgliÃ¨,2003,7,4,0,2585,196.633273,1,201,TO,13.1463,both,0.0,270.793037,154.738878,0.0,0.304268
3,1001,AgliÃ¨,2004,13,9,0,2585,196.633273,1,201,TO,13.1463,both,0.0,502.901354,348.162476,0.0,0.684603
4,1001,AgliÃ¨,2005,2,2,0,2585,196.633273,1,201,TO,13.1463,both,0.0,77.369439,77.369439,0.0,0.152134


Verific il dataset per capire se il dato è uniforme per comuni e date

In [26]:
anni = m.groupby("REF_AREA")["TIME_PERIOD"].nunique()

print(anni.value_counts().sort_index())

TIME_PERIOD
1        6
4        3
5        4
6       29
7       15
8       12
9       27
10       8
11      22
14       9
15     107
21       1
22       1
23    1443
24    5828
Name: count, dtype: int64


Noto che per alcuni comuni il dato non è analizzato per tutti i 24 anni, cerco di capir gli anni incompleti

In [27]:
tutti = set(m["TIME_PERIOD"].unique())

c23 = anni[anni == 23].index
mancante = (m[m["REF_AREA"].isin(c23)]
            .groupby("REF_AREA")["TIME_PERIOD"]
            .apply(lambda s: list(tutti - set(s))[0]))

print(mancante.value_counts().sort_index())

TIME_PERIOD
2024    1443
Name: count, dtype: int64


il 2024 on copre tutti i comuni, ora analizzo quei 107 comuni che hanno solo 15 anni

In [28]:
c15 = anni[anni == 15].index
print(m[m["REF_AREA"].isin(c15)].drop_duplicates("REF_AREA")["Codice Regione"].value_counts())

Codice Regione
3     55
11    34
16    10
8      5
4      2
6      1
Name: count, dtype: int64


In [29]:
c15 = anni[anni == 15].index

rng15 = (m[m["REF_AREA"].isin(c15)]
         .groupby("REF_AREA")["TIME_PERIOD"]
         .agg(["min", "max", "nunique"]))
rng15["buchi"] = (rng15["max"] - rng15["min"] + 1) - rng15["nunique"]

print(rng15.groupby(["min", "max"]).size())
print((rng15["buchi"] > 0).sum())   # quanti hanno buchi in mezzo

min   max 
2009  2023      1
2010  2024    106
dtype: int64
0


noto che per quei comuni il dato è mancante prima del 2010

In [30]:
nomi = (m[m["REF_AREA"].isin(c15)]
        .drop_duplicates("REF_AREA")[["REF_AREA", "Comune", "Codice Regione"]]
        .merge(rng15, left_on="REF_AREA", right_index=True))
print(nomi.sort_values(["Codice Regione", "Comune"]).head(30))

        REF_AREA               Comune  Codice Regione   min   max  nunique  \
174244    108001       Agrate Brianza               3  2010  2024       15   
174259    108002             Aicurzio               3  2010  2024       15   
174274    108003              Albiate               3  2010  2024       15   
174289    108004               Arcore               3  2010  2024       15   
174304    108005           Barlassina               3  2010  2024       15   
174319    108006             Bellusco               3  2010  2024       15   
174334    108007          Bernareggio               3  2010  2024       15   
174349    108008    Besana in Brianza               3  2010  2024       15   
174364    108009             Biassono               3  2010  2024       15   
174379    108010     Bovisio-Masciago               3  2010  2024       15   
174394    108011              Briosco               3  2010  2024       15   
174409    108012            Brugherio               3  2010  202

In [36]:
c15 = anni[anni == 15].index
print(m[m["REF_AREA"].isin(c15)]
      .drop_duplicates("REF_AREA")["Sigla automobilistica"]
      .value_counts())

Sigla automobilistica
MB    55
FM    34
BT    10
RN     5
TN     2
UD     1
Name: count, dtype: int64


Dopo una ricerca trovo i comuni che sono nati dopo il 2009

In [38]:
fin = m[m["TIME_PERIOD"].between(2010, 2023)]
n_anni = fin.groupby("REF_AREA")["TIME_PERIOD"].nunique()
completi = n_anni[n_anni == 14].index

tot = m["REF_AREA"].nunique()
print(len(completi), tot, round(len(completi) / tot * 100, 1))


7388 7515 98.3


Decido se posso stringere il range di date ed escludere eventualmente alcuni comuni, quanto impatta sull dato


In [39]:
u = m.drop_duplicates("REF_AREA")
quota = u.loc[u["REF_AREA"].isin(completi), "Popolazione residente"].sum() / u["Popolazione residente"].sum()
print(round(quota * 100, 1))

98.8


In [40]:
m = m[m["TIME_PERIOD"].between(2010, 2023)].copy()
print(m["TIME_PERIOD"].min(), m["TIME_PERIOD"].max())   # 2010 2023
print(m["TIME_PERIOD"].nunique())                       # 14

2010 2023
14


In [41]:
n_anni = m.groupby("REF_AREA")["TIME_PERIOD"].nunique()
completi = n_anni[n_anni == 14].index

print(len(completi), m["REF_AREA"].nunique())           # quanti restano e quanti c'erano
m = m[m["REF_AREA"].isin(completi)].copy()

7388 7511


In [42]:
print(len(m), m["REF_AREA"].nunique() * 14)   # i due numeri devono coincidere
print(m[["morti", "feriti", "incidenti"]].isna().sum())   # nessun NaN atteso

103432 103432
morti        0
feriti       0
incidenti    0
dtype: int64


In [43]:
print(m[["Popolazione residente", "Superficie (Kmq)", "densita"]].isna().sum())
print((m["Popolazione residente"] <= 0).sum(), (m["Superficie (Kmq)"] <= 0).sum())

Popolazione residente    0
Superficie (Kmq)         0
densita                  0
dtype: int64
0 0


Ho un dato valido al 98.8 percento di copertura, lo tengo ed inserisco le voci per l'analisi

In [44]:
m["morti_100k"]     = m["morti"]     / m["Popolazione residente"] * 100_000
m["feriti_100k"]    = m["feriti"]    / m["Popolazione residente"] * 100_000
m["incidenti_100k"] = m["incidenti"] / m["Popolazione residente"] * 100_000
m["incidenti_kmq"]  = m["incidenti"] / m["Superficie (Kmq)"]

In [45]:
m["indice_mortalita"] = (m["morti"] / m["incidenti"].where(m["incidenti"] > 0)) * 1000

In [46]:
m.head()

,REF_AREA,Comune,TIME_PERIOD,feriti,incidenti,morti,Popolazione residente,densita,Codice Regione,Codice Provincia/Uts,Sigla automobilistica,Superficie (Kmq),_merge,morti_100k,feriti_100k,incidenti_100k,indice_mortalita,incidenti_per_kmq,incidenti_kmq
9,1001,AgliÃ¨,2010,14,7,0,2585,196.633273,1,201,TO,13.1463,both,0.0,541.586074,270.793037,0.0,0.532469,0.532469
10,1001,AgliÃ¨,2011,6,4,0,2585,196.633273,1,201,TO,13.1463,both,0.0,232.108317,154.738878,0.0,0.304268,0.304268
11,1001,AgliÃ¨,2012,8,3,0,2585,196.633273,1,201,TO,13.1463,both,0.0,309.477756,116.054159,0.0,0.228201,0.228201
12,1001,AgliÃ¨,2013,5,4,0,2585,196.633273,1,201,TO,13.1463,both,0.0,193.423598,154.738878,0.0,0.304268,0.304268
13,1001,AgliÃ¨,2014,11,6,0,2585,196.633273,1,201,TO,13.1463,both,0.0,425.531915,232.108317,0.0,0.456402,0.456402


In [47]:
print(m.dtypes)

REF_AREA                    int64
Comune                        str
TIME_PERIOD                 int64
feriti                      Int64
incidenti                   Int64
morti                       Int64
Popolazione residente       int64
densita                   float64
Codice Regione              int64
Codice Provincia/Uts        int64
Sigla automobilistica         str
Superficie (Kmq)          float64
_merge                   category
morti_100k                Float64
feriti_100k               Float64
incidenti_100k            Float64
indice_mortalita          Float64
incidenti_per_kmq         Float64
incidenti_kmq             Float64
dtype: object


In [48]:
m = m.drop(columns=["_merge"], errors="ignore")
print(m.columns.tolist())

['REF_AREA', 'Comune', 'TIME_PERIOD', 'feriti', 'incidenti', 'morti', 'Popolazione residente', 'densita', 'Codice Regione', 'Codice Provincia/Uts', 'Sigla automobilistica', 'Superficie (Kmq)', 'morti_100k', 'feriti_100k', 'incidenti_100k', 'indice_mortalita', 'incidenti_per_kmq', 'incidenti_kmq']


In [49]:
m.to_csv("incidenti_2010_2023.csv", index=False, encoding="utf-8")

In [1]:
import requests

api_url = "https://situas.istat.it/ShibO2Module/api/Report/Spool/2026-10-08/68?&PageNum=1&PageSize=100"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Referer": "https://situas.istat.it/web/",
    "Content-Type": "application/json",
}

r = requests.post(api_url, headers=headers, json={})
print(r.status_code)
print(r.text[:1000])

200
{"label":["Codice Ripartizione geografica","Codice Regione","Regione","Ripartizione geografica","Flag tipo regione","Tipo regione","Codice fiscale","Codice NUTS2 2024"],"totalRows":20,"body":[{"COD_RIP":1,"COD_REG":"01","DEN_REG":"Piemonte","DEN_RIP":"Nord-ovest","TIPO_REG":"1","DESC_TIPO_REG":"Regione a Statuto ordinario","COD_REG_FISCALE":"80087670016","COD_NUTS2_2024":"ITC1"},{"COD_RIP":1,"COD_REG":"02","DEN_REG":"Valle d'Aosta/Vallée d'Aoste","DEN_RIP":"Nord-ovest","TIPO_REG":"2","DESC_TIPO_REG":"Regione a Statuto speciale","COD_REG_FISCALE":"80002270074","COD_NUTS2_2024":"ITC2"},{"COD_RIP":1,"COD_REG":"03","DEN_REG":"Lombardia","DEN_RIP":"Nord-ovest","TIPO_REG":"1","DESC_TIPO_REG":"Regione a Statuto ordinario","COD_REG_FISCALE":"80050050154","COD_NUTS2_2024":"ITC4"},{"COD_RIP":2,"COD_REG":"04","DEN_REG":"Trentino-Alto Adige/Südtirol","DEN_RIP":"Nord-est","TIPO_REG":"2","DESC_TIPO_REG":"Regione a Statuto speciale","COD_REG_FISCALE":"80003690221"},{"COD_RIP":2,"COD_REG":"05","DE

In [2]:
import json

dati = r.json()
print(type(dati))
print(json.dumps(dati, indent=2, ensure_ascii=False)[:3000])

<class 'dict'>
{
  "label": [
    "Codice Ripartizione geografica",
    "Codice Regione",
    "Regione",
    "Ripartizione geografica",
    "Flag tipo regione",
    "Tipo regione",
    "Codice fiscale",
    "Codice NUTS2 2024"
  ],
  "totalRows": 20,
  "body": [
    {
      "COD_RIP": 1,
      "COD_REG": "01",
      "DEN_REG": "Piemonte",
      "DEN_RIP": "Nord-ovest",
      "TIPO_REG": "1",
      "DESC_TIPO_REG": "Regione a Statuto ordinario",
      "COD_REG_FISCALE": "80087670016",
      "COD_NUTS2_2024": "ITC1"
    },
    {
      "COD_RIP": 1,
      "COD_REG": "02",
      "DEN_REG": "Valle d'Aosta/Vallée d'Aoste",
      "DEN_RIP": "Nord-ovest",
      "TIPO_REG": "2",
      "DESC_TIPO_REG": "Regione a Statuto speciale",
      "COD_REG_FISCALE": "80002270074",
      "COD_NUTS2_2024": "ITC2"
    },
    {
      "COD_RIP": 1,
      "COD_REG": "03",
      "DEN_REG": "Lombardia",
      "DEN_RIP": "Nord-ovest",
      "TIPO_REG": "1",
      "DESC_TIPO_REG": "Regione a Statuto ordinario",
   

In [3]:
import pandas as pd

df_regioni = pd.DataFrame(dati["body"])[["COD_REG", "DEN_REG"]]
df_regioni.columns = ["codice_regione", "regione"]

print(len(df_regioni))   
print(df_regioni)

20
   codice_regione                       regione
0              01                      Piemonte
1              02  Valle d'Aosta/Vallée d'Aoste
2              03                     Lombardia
3              04  Trentino-Alto Adige/Südtirol
4              05                        Veneto
5              06         Friuli-Venezia Giulia
6              07                       Liguria
7              08                Emilia-Romagna
8              09                       Toscana
9              10                        Umbria
10             11                        Marche
11             12                         Lazio
12             13                       Abruzzo
13             14                        Molise
14             15                      Campania
15             16                        Puglia
16             17                    Basilicata
17             18                      Calabria
18             19                       Sicilia
19             20                    

In [4]:
import pandas as pd

df_regioni = pd.DataFrame(dati["body"])[["COD_REG", "DEN_REG"]]
df_regioni.columns = ["codice_regione", "regione"]

print(len(df_regioni))   
print(df_regioni)

20
   codice_regione                       regione
0              01                      Piemonte
1              02  Valle d'Aosta/Vallée d'Aoste
2              03                     Lombardia
3              04  Trentino-Alto Adige/Südtirol
4              05                        Veneto
5              06         Friuli-Venezia Giulia
6              07                       Liguria
7              08                Emilia-Romagna
8              09                       Toscana
9              10                        Umbria
10             11                        Marche
11             12                         Lazio
12             13                       Abruzzo
13             14                        Molise
14             15                      Campania
15             16                        Puglia
16             17                    Basilicata
17             18                      Calabria
18             19                       Sicilia
19             20                    

In [ ]:
import pandas as pd

df = pd.read_csv("incidenti_2010_2023.csv", encoding="utf-8")

# 1. tolgo le colonne regione duplicate e quella ridondante
df = df.drop(columns=["Regione_x", "Regione_y", "Regione", "incidenti_per_kmq"], errors="ignore")

# 2. correggo i nomi con accenti rovinati
df["Comune"] = df["Comune"].str.encode("latin-1").str.decode("utf-8")

# 3. aggiungo la regione 
regioni_df = df_regioni.copy()
regioni_df["codice_regione"] = regioni_df["codice_regione"].astype(int)
regioni_df = regioni_df.rename(columns={"regione": "Regione"})

df = df.merge(regioni_df, left_on="Codice Regione", right_on="codice_regione", how="left")
df = df.drop(columns="codice_regione")

print(df["Regione"].isna().sum())   
df.to_csv("incidenti_2010_2023_regioni.csv", index=False, encoding="utf-8")

0
